## **Fire Heatmap of South America**
-----
#### SDS 210 - Programming with Spatial Data

*Author: Isabelle Bartholet*

*Date: May 2026*

### **Research Question**
* How has the fire distribution in South America varied over the past five days?
* Are there any patterns identifiable?


### **Content**

1. Load important packages 
2. Call map key via function check_map_key()
3. Fetch the Data via function fetch_data()
4. Convert Data to gdf
5. Create Heatmap


------

#### **Import Libraries and Packages**

In [1]:
import requests
import pandas as pd
import geopandas as gpd
import time
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import folium
import contextily
import cmcrameri
import branca.colormap as cm
import numpy as np



from cartopy import crs as ccrs
from geodatasets import get_path


In [2]:
from config import MAP_KEY
from access_mapkey import check_map_key

#### **Access Map Key**

In [3]:
check_map_key(MAP_KEY)

transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object


{'transaction_limit': 5000,
 'current_transactions': 0,
 'transaction_interval': '10 minutes'}

#### **Import Data for the Last Five Days**

In [5]:
from FetchFireData2 import fetch_data
from config import MAP_KEY

df_fire = fetch_data(MAP_KEY)

# check length of df_fire, to see whether it worked correctly
print(f"There are {len(df_fire)} rows in this data frame.")

# quickly check dataframe by using head
df_fire.head(6)


Request successful


 Data download successful! 7177 fires found.
There are 7177 rows in this data frame.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,-5.44360,-36.79174,303.93,0.41,0.61,2026-05-14,313,N20,VIIRS,n,2.0NRT,285.17,1.14,N
1,-12.40172,-38.34732,302.24,0.40,0.60,2026-05-14,315,N20,VIIRS,n,2.0NRT,288.83,1.39,N
2,-12.39697,-38.34255,309.97,0.40,0.60,2026-05-14,315,N20,VIIRS,n,2.0NRT,289.85,1.18,N
3,-12.39624,-38.34624,305.83,0.40,0.60,2026-05-14,315,N20,VIIRS,n,2.0NRT,289.12,1.18,N
4,-9.46632,-40.34198,347.44,0.61,0.71,2026-05-14,315,N20,VIIRS,n,2.0NRT,288.26,31.16,N
5,-9.46524,-40.34753,315.09,0.61,0.71,2026-05-14,315,N20,VIIRS,n,2.0NRT,287.40,43.93,N


The data is stored in a dataframe called df_fire.

#### **Convert Fire Dataframe into GDF and Check for NAs**


In [6]:
# create gdf 
gdf_fire = gpd.GeoDataFrame(
    df_fire, 
    geometry=gpd.points_from_xy(
        df_fire.longitude, df_fire.latitude),
        
        crs="EPSG:4326")

# check, to see if gdf returns the same information as the df above 
gdf_fire.head(6)


# check for NAs
gdf_fire.info()
print(f"There are no NAs in this gdf.") ## noch ändern zu etwas, was mit gdf_fire.info() zusammenhängt


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 7177 entries, 0 to 7176
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   latitude    7177 non-null   float64 
 1   longitude   7177 non-null   float64 
 2   bright_ti4  7177 non-null   float64 
 3   scan        7177 non-null   float64 
 4   track       7177 non-null   float64 
 5   acq_date    7177 non-null   str     
 6   acq_time    7177 non-null   int64   
 7   satellite   7177 non-null   str     
 8   instrument  7177 non-null   str     
 9   confidence  7177 non-null   str     
 10  version     7177 non-null   str     
 11  bright_ti5  7177 non-null   float64 
 12  frp         7177 non-null   float64 
 13  daynight    7177 non-null   str     
 14  geometry    7177 non-null   geometry
dtypes: float64(7), geometry(1), int64(1), str(6)
memory usage: 1023.4 KB
There are no NAs in this gdf.


#### **Filter Data for Confindence Values without low confidence** 
The confidence attribute identifies the quality of every hotspot/fire pixel. It accounts for sun glint contamination during the day. Areas with low confidence (l) are associated with areas of sun glint. Nominal confidence pixels (n) are free of potential sun glint contamination. Pixels showing high confidence (h) are indicative for saturated day or nighttime pixels.

l: low

n: nominal

h: high

In [7]:
gdf_fire_reduced = gdf_fire[(gdf_fire["confidence"]!="l")].copy()
gdf_fire_reduced.head(10)

# check if no low confidence fires are included

gdf_fire_reduced["confidence"].unique()

#other method
#gdf_fire_reduced[gdf_fire_reduced["confidence"] == "l"]

<ArrowStringArray>
['n', 'h']
Length: 2, dtype: str

#### **Heatmap**

Time animated heatmap including a scale and zoom control.

In [ ]:
from folium.plugins import HeatMapWithTime
from scipy.spatial import cKDTree

## convert dates into a specialized time-aware objects (datetime64)
gdf_fire_reduced["acq_date"] = pd.to_datetime(gdf_fire_reduced["acq_date"])


## sort days, creates a list with date object, which will be looped through
unique_days = sorted(gdf_fire_reduced["acq_date"].dt.date.unique())


# gdf_fire_reduced["acq_date"]: access "aqc_date" column in gdf_fire_reduced -> returns Pandas-Series
# dt.: to access time/date values (from datetime)
# .date: removes hours from timeobject
# .unique: takes unique values (maybe unnecessary?)
# sorted(): sorts it in chronological order -> result is a sorted list




## create heatmap data, so HeatMapWithTime can use / work with it
# heatmap expects a list like this
# data =[
# data = [
 #   [[lat, lon, weight], [lat, lon, weight], ...],
#  The outer list corresponds to the various time steps in sequential order


data = [] #create empty list; here all the days will be saved


for day in unique_days:
    subset = gdf_fire_reduced[gdf_fire_reduced["acq_date"].dt.date == day] # compare "acq_date" column with day (from unique_days above); is saved as time
    day_data = [
        [float(row.geometry.y), float(row.geometry.x)]
         #float() to create normal Python float values, currently they are np.float64 dataypes (see .info() above)
         for _, row in subset.iterrows()
    ]

    data.append(day_data)

## create time index for timeline at the bottom of the map
time_index = [str(day) for day in unique_days]


## initialize basemap centered on South America, add a zoom control limit
min_lon, max_lon = -81.5, -35.0
min_lat, max_lat = -64.0, 15.5

base_map = folium.Map(
    max_bounds = True,
    location = [8, -69],
    zoom_start = 3,
    tiles = "CartoDB DarkMatter", #or better with CartoDB Positron
    control_scale = True,
    min_lat = min_lat,
    max_lat = max_lat,
    min_lon = min_lon,
    max_lon = max_lon

)

## heatmap
# 0-1 as threshold values for pixel intensity
HeatMapWithTime(
    data,
    index=time_index,
    name = "Heatmap of Last 5 Days",
    auto_play = True,
    radius = 10,
    blur = 0.8,
    min_opacity= 0.3,
    gradient = {
        0.0: "#0000ff", # isolated value
        0.5: "#f9e107", #medium density
        1.0: "#ff0101", # high density
    }
).add_to(base_map)


## html legend
legend_html = """
<div style = "position: fixed;
bottom: 50px; 
right: 50px;
width: 170px; 
height: 100px;
background-color: lightgrey;
z-index:9999;
font-size:14px;
padding: 10px;
border: 2px solid grey;
">

<b>Fire Density</b><br>
<i style="background:#0000ff;width:20px;height:10px;float:left;margin-right:8px;"></i> Low Density<br>

<i style="background:#f9e107;width:20px;height:10px;float:left;margin-right:8px;"></i> Medium Density<br>

<i style="background:#ff0101;width:20px;height:10px;float:left;margin-right:8px;"></i> High Density

</div>
"""

## add legend to the map
base_map.get_root().html.add_child(folium.Element(legend_html))


## save map because I cannot open it in VS Code directly
base_map.save("Time_Animated_Heatmap.html")

base_map




[[[-5.4436, -36.79174],
  [-12.40172, -38.34732],
  [-12.39697, -38.34255],
  [-12.39624, -38.34624],
  [-9.46632, -40.34198],
  [-9.46524, -40.34753],
  [-9.4609, -40.33513],
  [-9.45982, -40.34066],
  [-9.45874, -40.34621],
  [-24.54681, -42.47359],
  [-24.54569, -42.46856],
  [-24.54473, -42.47261],
  [-24.54377, -42.47668],
  [-24.54223, -42.46792],
  [-24.54129, -42.47199],
  [-24.54035, -42.47606],
  [-20.25082, -40.23106],
  [-20.25005, -40.23456],
  [-20.24928, -40.23807],
  [-20.24851, -40.24158],
  [-20.24774, -40.24513],
  [-20.24322, -40.24027],
  [-20.24244, -40.24381],
  [-20.24167, -40.24734],
  [-20.24043, -40.23886],
  [-20.23968, -40.24241],
  [-20.23892, -40.24597],
  [-20.23793, -40.23894],
  [-20.23715, -40.24248],
  [-20.23664, -40.23045],
  [-20.23589, -40.23397],
  [-20.23419, -40.23058],
  [-20.23341, -40.23409],
  [-20.23135, -40.22913],
  [-20.2306, -40.23264],
  [-20.22985, -40.23615],
  [-20.22812, -40.23276],
  [-17.37095, -42.6974],
  [-17.36443, -42.6960

### **Answer Research Question**

*1. How has the fire distribution in South America varied over the past five days?*

Type answer here

*2. Are there any patterns identifiable?*

Type answer here

